# Prep: QLoRA fine-tuning of BLOOMZ-7b1-mt for news sentiment

Instruction-style prompts, 4-bit NF4 quantisation, LoRA via `trl.SFTTrainer`, and label scoring by log-probabilities (not fully run).

In [ ]:
DATA_DIR = "../data"

In [1]:
!pip install -q -U trl transformers accelerate sentencepiece git+https://github.com/huggingface/peft.git
!pip install -q -U datasets bitsandbytes einops scipy

In [3]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '7'

In [5]:
import pandas as pd
import numpy as np
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import torch
from tqdm import tqdm

In [10]:
df = pd.read_csv(f"{DATA_DIR}/gemastik_22.csv")
df.head()

,id,sumber,kodekat,kodesubkat,kategori,subkategori,katakunci,tanggal,judul_berita,konten_berita,nama_tokoh,jabatan,organisasi,lokasi,alias,kutipan,sentimen_kutipan,sentimen_berita
0,568576e9e95,Detik,Q,Q,Jasa Kesehatan dan Kegiatan Sosial Swasta,Jasa Kesehatan dan Kegiatan Sosial Swasta,Jumlah pasien,2021-06-17,"BOR Sudah 89 Persen, RSUD Soreang Bandung Siap...",Bandung -\n\nSatuan Tugas Penanganan COVID-19 ...,[],['Bupati Bandung'],"['Bed Occupancy Ratio (BOR)', 'Pemda', 'BOR']","['Bandung', 'Kabupaten Bandung', 'Kota Bandung']",[],"['mengatakan, telah mengadakan rapat dan memut...",Netral,Netral
1,80f931fe95e,Detik,D,D2,Pengadaan Listrik dan Gas,Pengadaan Gas dan Produksi Es,Jaringan Gas,2021-03-18,"Honda CB650R Indonesia Dapat Penyegaran, Harga...",Jakarta -\n\nAstra Honda Motor (AHM) resmi men...,"['Biasa Mejeng', 'Resmi Dijual']",[],"['Big', 'LED']","['Jakarta', 'Indonesia']",[],"['""Rasa bangga itu menyempurnakan gaya hidup m...",Positif,Positif
2,1bd2af6ef23,Antara,C,C8,Industri Pengolahan,"Industri Kimia, Farmasi dan Obat Tradisional",farmasi,2021-02-23,Vaksin COVID-19 Pfizer disetujui satu dari tig...,Seoul (ANTARA) - Panel ahli pertama dari tiga ...,['Mulyo Sunyoto'],[],"['Universitas Oxford', 'BioNTech']","['Korea Selatan', 'Amerika Serikat', 'Korsel',...",['¬©'],[],Netral,Positif
3,ac2b55af24c,Detik,A,A2,"Pertanian, Kehutanan, dan Perikanan",Kehutanan dan Penebangan Kayu,bambu,2021-01-26,"TPU Srengseng Sawah Penuh, DKI Rujuk Jenazah C...","Jakarta - TPU Srengseng Sawah, Jakarta Selatan...",['Ivan Nurcahyo'],[],"['Pemprov DKI', 'Dinas Pertamanan']","['Jakarta', 'Jakarta Selatan', 'DKI Jakarta', ...",[],"['""Kita sebutnya TPU Bambu Wulung, sebenarnya ...",Positif,Positif
4,3f50747a1d4,Antara,Q,Q,Jasa Kesehatan dan Kegiatan Sosial Swasta,Jasa Kesehatan dan Kegiatan Sosial Swasta,wabah penyakit,2021-05-04,Media Korea Utara sebut vaksin COVID-19 bukan ...,Seoul (ANTARA) - Media pemerintah Korea Utara ...,['Edwin Salvador'],[],['Partai Buruh'],"['Korea Utara', 'Korea Selatan', 'China', 'Kor...",['¬©'],['mengatakan wabah di sana tidak dapat dikesam...,Netral,Netral


In [11]:
LABELS_TO_EN = {
    'Positif': 'Positive',
    'Netral': 'Neutral',
    'Negatif': 'Negative'
}
df['train_prompt'] = "### News:\n" + df['judul_berita'] + "\n" + df['konten_berita'] \
                + "'\n\n### Instruction:\nWould you rate the previous news as Positive, Neutral or Negative?" + "'\n\n### Answer:\n" + df['sentimen_berita'].map(LABELS_TO_EN)
df['train_prompt'].head()

0    ### News:\nBOR Sudah 89 Persen, RSUD Soreang B...
1    ### News:\nHonda CB650R Indonesia Dapat Penyeg...
2    ### News:\nVaksin COVID-19 Pfizer disetujui sa...
3    ### News:\nTPU Srengseng Sawah Penuh, DKI Ruju...
4    ### News:\nMedia Korea Utara sebut vaksin COVI...
Name: train_prompt, dtype: object

In [12]:
print(df['train_prompt'][0])

### News:
BOR Sudah 89 Persen, RSUD Soreang Bandung Siapkan 100 Kasur
Bandung -

Satuan Tugas Penanganan COVID-19 Kabupaten Bandung memutuskan akan menggunakan RSUD Soreang menjadi lokasi penanganan pasien COVID-19. Sebanyak 100 kasur akan disiapkan di RSUD Soreang.

Sebelumnya, Kabupaten Bandung masuk dalam zona risiko tinggi atau zona merah. Per tanggal 16 Juni 2021, Bed Occupancy Ratio (BOR) atau keterisian ruang isolasi mencapai 89 persen.

Hal tersebut diungkapkan oleh Bupati Bandung Dadang Supriatna. Ia mengatakan, telah mengadakan rapat dan memutuskan untuk menggunakan RSUD Soreang dijadikan rumah sakit rujukan bagi pasien COVID-19.

"Kita sudah instruksikan dan rapat evaluasi untuk bed, karena RSUD Soreang lama sudah ada pemindahan dan ijin sudah hampir selesai. Maka kami sudah sepakat akan menambah 100 bed isolasi dalam membantu masyarakat," ujar Bupati Bandung Dadang Supriatna, di Komplek Pemda Bandung, Kamis (1762021).

Selain di RSUD Soreang, sejumlah rumah sakit rujukan CO

In [13]:
train_prompt = df[['train_prompt']]

X_train_prompt, X_val_prompt = train_test_split(train_prompt, test_size=0.2, random_state=42)

X_train_prompt.shape, X_val_prompt.shape

((4231, 1), (1058, 1))

In [14]:
X_train_prompt

,train_prompt
1448,### News:\nCerita Perajin Batik Tasikmalaya Ra...
3492,### News:\nYang Perlu Kamu Tahu buat Investasi...
3296,### News:\nAjakan Ma'ruf Amin ke Lansia Pastik...
1305,### News:\nTFRIC-19 ciptakan RI-GHA COVID-19 d...
3887,### News:\nWisatawan TMII stabil di kisaran 11...
...,...
3092,### News:\nPemprov DKI Tambah TPU Khusus COVID...
3772,"### News:\nTercepat di Dunia, India Salurkan 1..."
5191,### News:\nKerugian bank global mencapai 6 mil...
5226,### News:\nVirus Corona Bikin Perayaan Tahun B...


In [15]:
tds = Dataset.from_dict({
    "text": X_train_prompt['train_prompt']
})
vds = Dataset.from_dict({
    "text": X_val_prompt['train_prompt']
})

ds = DatasetDict()

ds['train'] = tds
ds['validation'] = vds

ds

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 4231
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 1058
    })
})

In [21]:
pip install -i https://test.pypi.org/simple/ bitsandbytes

In [22]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, AutoTokenizer

model_name = "bigscience/bloomz-7b1-mt"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    trust_remote_code=True
)
model.config.use_cache = False

ImportError: ignored

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:
from peft import LoraConfig, get_peft_model

lora_alpha = 16
lora_dropout = 0.1
lora_r = 64

peft_config = LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r=lora_r,
    bias="none",
    task_type="CAUSAL_LM"
)

In [ ]:
from transformers import TrainingArguments

output_dir = "./results_bloomz_qlora"
per_device_train_batch_size = 1
gradient_accumulation_steps = 1
optim = "paged_adamw_32bit"
save_steps = 100
logging_steps = 10
learning_rate = 4e-3
max_grad_norm = 0.3
max_steps = -1
warmup_ratio = 0.03
lr_scheduler_type = "constant"

training_arguments = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim=optim,
    save_steps=save_steps,
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    fp16=True,
    max_grad_norm=max_grad_norm,
    max_steps=max_steps,
    warmup_ratio=warmup_ratio,
    group_by_length=True,
    lr_scheduler_type=lr_scheduler_type,
    num_train_epochs=1,
)

In [ ]:
from trl import SFTTrainer

max_seq_length = 64

trainer = SFTTrainer(
    model=model,
    train_dataset=ds['train'],
    eval_dataset=ds['validation'],
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    tokenizer=tokenizer,
    args=training_arguments,
)

In [ ]:
for name, module in trainer.model.named_modules():
    if "norm" in name:
        module = module.to(torch.float32)

In [ ]:
trainer.train()

In [ ]:
model_to_save = trainer.model.module if hasattr(trainer.model, 'module') else trainer.model  # Take care of distributed/parallel training
model_to_save.save_pretrained("bloomz_qlora")

In [ ]:
lora_config = LoraConfig.from_pretrained('bloomz_qlora')
model = AutoModelForCausalLM.from_pretrained(
    'bloomz_qlora',
    quantization_config=bnb_config,
    trust_remote_code=True
)
model = get_peft_model(model, lora_config)

In [ ]:
# function to fully generate output text
def generate_text(input_prompt, min_length=0, max_length=100):
  input_prompt = "### News:\n" + input_prompt \
                + "'\n\n### Instruction:\nWould you rate the previous news as Positive, Neutral or Negative?" + "'\n\n### Answer:\n"
  encoded_input = tokenizer(input_prompt, return_tensors='pt').to('cuda')
  output = model.generate(**encoded_input, min_length=min_length, max_length=max_length, repetition_penalty=1.5)

  print(tokenizer.batch_decode(output, skip_special_tokens=True)[0])

import numpy, torch
import torch.nn.functional as F

# instead of free-text generation, we simply get the probability of each possible label, then pick the most likely
def get_logprobs(prompt, device = 'cuda'):
  inputs = tokenizer(prompt, return_tensors="pt").to(device)
  input_ids, output_ids = inputs["input_ids"], inputs["input_ids"][:, 1:]
  outputs = model(**inputs, labels=input_ids)
  logits = outputs.logits
  logprobs = torch.gather(F.log_softmax(logits, dim=2), 2, output_ids.unsqueeze(2))
  return logprobs.sum().detach().cpu()

def classify_with_LLM(input_prompt):
  classes = ['Positive', 'Negative', 'Neutral']
  input_prompt = "### News:\n" + input_prompt \
                + "'\n\n### Instruction:\nWould you rate the previous news as Positive, Neutral or Negative?" + "'\n\n### Answer:\n"
  logprobs = [get_logprobs(input_prompt + c) for c in classes]

  return classes[numpy.argmax(logprobs)]

In [ ]:
print(classify_with_LLM("""Kamu jahat"""))
print(classify_with_LLM("""baik banget deh kamu"""))
print(classify_with_LLM("""besok gw mau senam, lu mau ikut ga"""))

In [ ]:
LABELS_TO_ID = {
    'Positive': 'Positif',
    'Negative': 'Negatif',
    'Neutral': 'Netral'
}

y_val_pred = []
for prompt in tqdm(X_val_prompt.train_prompt.values):
    y_val_pred.append(LABELS_TO_ID[classify_with_LLM(prompt)])
y_val_pred[:10]

In [ ]:
print(classification_report(df.loc[X_val_prompt.index, 'sentimen_berita'].values.tolist(), y_val_pred))

In [ ]:
torch.cuda.empty_cache()

In [ ]:
input_prompt = """"
### News:
"UNI Emirat Arab (UEA) menutup sementara penerbangan dari India mulai Minggu 25 April 2021 hingga 10 hari ke depan, menyusul melonjaknya kasus COVID-19 di Negeri Bollywood.

Imbas kebijakan ini, sejumlah ekspatriat terjebak di India dan tak bisa kembali ke UEA. Mereka terpaksa menunggu sampai penerbangan dibuka lagi.

Beberapa penduduk UEA ke India untuk liburan tahunan. Ada juga yang pergi menjenguk anggota keluarga yang sakit atau untuk keberluan pekerjaan bank dan properti.

Baca juga: Pengakuan Para Ekspatriat yang Berhasil Keluar India Jelang Ditutupnya Penerbangan

Namun, petugas kesehatan dan orang tua dari anak-anak yang belajar di India telah mengimbau pemerintah UEA untuk membebaskan mereka dari skorsing penerbangan 10 hari, yang dimulai pada Minggu, 25 April.

Vikas Chopra, warga asal India yang sudah 20 tahun tinggal di Dubai, mengatakan putranya yang berusia 15 tahun, Kovid Chopra, seorang siswa sekolah asrama di Kodaikanal, Tamil Nadu, masih terjebak di India.

Sesi dia berakhir pada 27 April dan penerbangannya ke Dubai dikonfirmasi pada 28 April. Semua siswa di sekolahnya telah kembali ke rumah masing-masing, tetapi dia terjebak sekarang.

Chopra mengatakan keluarga besarnya tinggal di Delhi dan mengingat situasi saat ini di ibu kota India, dia tidak ingin mengirim putranya ke sana.

Karena Kodaikanal bukan kota metro, keluarganya juga tidak bisa segera mengatur kepulangannya ke Dubai dalam waktu singkat.

â€šÃ„ÃºUntuk saat ini, setelah sekolah tutup, dia bepergian dengan teman sekelasnya ke Bangalore. Dia sudah mendapat persetujuan GDRFA yang berlaku hingga 13 Mei,â€šÃ„Ã¹ kata Chopra seperti dilansir dari Khaleej Times, Senin (2642021).

Baca juga: Uni Emirat Arab Kaji Pembatasan bagi Warga Belum Divaksin Covid-19

Keluarganya berharap dia bisa segera terbang kembali ke orang tuanya di Dubai.

Raheesh Babu, manajer umum Musafir.com mengatakan kepada Khaleej Times, â€šÃ„ÃºBanyak keluarga yang terlantar di rumah, terutama di Kerala. Karena sekolah tutup untuk liburan, mereka (keluarga) melakukan perjalanan untuk liburan dengan harapan bisa kembali pada akhir April. Sekarang, semua yang terdampar berharap semuanya akan kembali normal setelah sepuluh hari. ""

Lihat juga: Salam Kangen Dari Endo Untuk Lisa Dimasa Karantina

Babu mengatakan banyak keluarga tidak dapat kembali dalam waktu dua hari sejak pengumuman larangan.

â€šÃ„ÃºSaya seharusnya terbang pada 27 April ke Dubai dari Bangalore dan kembali bekerja keesokan harinya. Namun, dengan semua penerbangan ke UEA ditangguhkan, saya harus menjadwal ulang penerbangan saya ke minggu pertama bulan depan,â€šÃ„Ã¹ kata Dipti Das, seorang analis konten asal India yang bekerja di UEA.

""Saya tidak yakin apakah larangan itu akan dicabut saat itu. Syukurlah, kantor saya telah memberi saya pilihan untuk bekerja dari rumah hingga situasinya kembali normal. Terlepas dari situasi yang mengerikan di rumah, satu-satunya penghilang kesedihan adalah saya tidak kehilangan gaji karena saya bisa bekerja dari sini,â€šÃ„Ã¹ ujarnya.

""Saya hanya berharap semuanya kembali normal dengan cepat,"" katanya.

Beberapa petugas kesehatan dari UEA yang saat ini berada di India untuk liburan mereka juga telah meminta pemerintah UEA untuk membebaskan mereka dari penangguhan tersebut.

Susan Koshy, seorang perawat Sharjah yang terbang pulang ke India bersama suami dan putrinya dua minggu lalu sedang mengalami perubahan.

â€šÃ„ÃºIni adalah liburan pertama saya setelah dua tahun. Saya datang menemui orang tua saya yang sakit dan mengurus beberapa urusan keluarga. Saya harus kembali pada minggu kedua bulan Mei. Suamiku bisa bekerja dari rumah, tapi aku tidak bisa,â€šÃ„Ã¹ katanya.

Sebelumnya

1

2

Selanjutnya

*  Ekspatriat

*  Covid-19 di India

*  Covid-19

*  Penerbangan India

*  india

*  Uni Emirat Arab

MENARIK UNTUK ANDA

Antrian ramai menunggu beli Replika ROLEX diskon 90% hari ini

Cara Menghilangkan Papiloma secara Alami Dalam 24 Jam

Orang Kaya Ini Bingung Bagaimana Menghabiskan Uangnya

Diabetes Hilang Selamanya! Lakukan Ini sebelum Tidur

Seluruh Negara Kaget! Diabetes Mudah Diobati (Lihat di Sini)

Uang Selalu Datang Melimpah, jika Benda Ini Ada DirumahBabu mengatakan banyak keluarga tidak dapat kembali dalam waktu dua hari sejak pengumuman larangan.

â€šÃ„ÃºSaya seharusnya terbang pada 27 April ke Dubai dari Bangalore dan kembali bekerja keesokan harinya. Namun, dengan semua penerbangan ke UEA ditangguhkan, saya harus menjadwal ulang penerbangan saya ke minggu pertama bulan depan,â€šÃ„Ã¹ kata Dipti Das, seorang analis konten asal India yang bekerja di UEA.

""Saya tidak yakin apakah larangan itu akan dicabut saat itu. Syukurlah, kantor saya telah memberi saya pilihan untuk bekerja dari rumah hingga situasinya kembali normal. Terlepas dari situasi yang mengerikan di rumah, satu-satunya penghilang kesedihan adalah saya tidak kehilangan gaji karena saya bisa bekerja dari sini,â€šÃ„Ã¹ ujarnya.

""Saya hanya berharap semuanya kembali normal dengan cepat,"" katanya.

Beberapa petugas kesehatan dari UEA yang saat ini berada di India untuk liburan mereka juga telah meminta pemerintah UEA untuk membebaskan mereka dari penangguhan tersebut.

Susan Koshy, seorang perawat Sharjah yang terbang pulang ke India bersama suami dan putrinya dua minggu lalu sedang mengalami perubahan.

â€šÃ„ÃºIni adalah liburan pertama saya setelah dua tahun. Saya datang menemui orang tua saya yang sakit dan mengurus beberapa urusan keluarga. Saya harus kembali pada minggu kedua bulan Mei. Suamiku bisa bekerja dari rumah, tapi aku tidak bisa,â€šÃ„Ã¹ katanya.

Lihat juga: Mantap Jiwa! Air Production Record Mempersembahkan Cover Lagu Jangan Rubah Takdirku

(sal)

Sebelumnya

2

2

Wisatawa...

*  Ekspatriat

*  Covid-19 di India

*  Covid-19

*  Penerbangan India

*  india

*  Uni Emirat Arab

MENARIK UNTUK ANDA

Antrian ramai menunggu beli Replika ROLEX diskon 90% hari ini

Cara Menghilangkan Papiloma secara Alami Dalam 24 Jam

Orang Kaya Ini Bingung Bagaimana Menghabiskan Uangnya

Diabetes Hilang Selamanya! Lakukan Ini sebelum Tidur

Dokter Berkata: Tidak Perlu Minum Pil Diabetes Setiap Hari!

Wanita 42 Tahun dengan Wajah Bayi: Dia Lakukan Ini sebelum Tidur"

### Instruction:
Would you rate the previous news as Positive, Neutral or Negative?

### Answer:"""
encoded_input = tokenizer(input_prompt, return_tensors='pt').to('cuda')
output = model.generate(**encoded_input, min_length=0, max_length=100, repetition_penalty=1.5)

print(tokenizer.batch_decode(output, skip_special_tokens=True)[0])

In [ ]:
print(get_logprobs(input_prompt + " Positive"))
print(get_logprobs(input_prompt + " Neutral"))
print(get_logprobs(input_prompt + " Negative"))

numpy.argmax([get_logprobs(input_prompt + " Positive"), get_logprobs(input_prompt + " Neutral"), get_logprobs(input_prompt + " Negative")])